# Electricity carbon footprint workflow

This notebook reproduces the electricity carbon footprint calculations used in the repository across multiple input-output databases.

## Before you run it

1. Check `paths.yml` and make sure each database path points to a valid local dataset copy.
2. Review `database_properties.py` if you want to change years, versions, systems, or electricity labels.
3. Activate the MARIO "dev" branch where `Database.calc_ghg` is available.

## How to use this notebook

- Run the setup cell first.
- Then run only the database sections you need.
- Each section exports one or more CSV files to `export/`.
- Run the final merge step to build `export/combined_results.csv` in `kg/EUR`.

The helper logic for path resolution, reshaping, and export lives outside the notebook in `common.py`, so the notebook remains focused on the execution workflow.

In [1]:
import warnings
import pandas as pd
import mario

from common import load_config, db_path, emission_factors, export_efs, should_skip
from database_properties import properties

# Keep notebook output readable while preserving MARIO progress messages.
mario.set_log_verbosity('info')
warnings.filterwarnings('ignore')

# How to react when a target CSV already exists in export/:
#   'ask'       -> prompt y/N for each file
#   'skip'      -> always reuse the existing export
#   'overwrite' -> always recompute and replace the export
OVERWRITE_POLICY = 'overwrite'

# Load path templates from paths.yml. `base` is only populated when using the
# older shared-root configuration; with the current explicit-path layout it is None.
cfg, base = load_config()

## EXIOBASE monetary IOT (`ixi` and `pxp`)

This section loops over the EXIOBASE versions, years, and system variants listed in `database_properties.py`.
If a target dataset folder is missing, the notebook attempts to download it before parsing and exporting the electricity-sector results.

In [ ]:
p = properties['exiobase_monetary_iot']
name, table = p['name'], p['table']

for version in p['versions']:
    for year in p['years']:
        for system in p['systems']:
            # EXIOBASE archives are resolved from the configured path template.
            archive = f'IOT_{year}_{system}'
            folder = db_path(
                cfg, base, 'EXIOBASE',
                version=version, table=table, system=system, year=year, archive=archive,
            )
            print(f'\n=== {name} {version} {table} {system} {year} ===')

            # Respect the overwrite policy before doing any download or parsing work.
            if should_skip(name, table, version, year, system, policy=OVERWRITE_POLICY):
                continue

            # Download the dataset only when the expected folder is not already available.
            if not folder.exists():
                folder.parent.mkdir(parents=True, exist_ok=True)
                try:
                    mario.download_exiobase3(
                        path=str(folder.parent),
                        years=year,
                        system=system,
                        table=table,
                        version=version,
                    )
                except Exception as exc:
                    print(f'[skip] download failed: {exc}')
                    continue

            # Parse the database, aggregate GHGs, then keep only electricity sectors.
            try:
                db = mario.parse_exiobase(
                    path=str(folder),
                    table=table,
                    unit='Monetary',
                )
            except Exception as exc:
                print(f'[skip] parse failed: {type(exc).__name__}: {exc}')
                continue

            db.calc_ghg(profile='exiobase_monetary', inplace=True)
            efs = emission_factors(db, 'Sector', p['labels_list'][system])
            export_efs(efs, name, table, version, year, system)

## EORA26

This section parses the EORA26 release configured in `database_properties.py`, aggregates the GHG satellite accounts, and exports the electricity-related results for the selected year.

In [ ]:
p = properties['eora26']
name, table = p['name'], p['table']

for version in p['versions']:
    for year in p['years']:
        # EORA26 is addressed through the configured full path template.
        path = db_path(cfg, base, 'EORA26', year=year)
        print(f'\n=== {name} {version} {table} {year} ===')

        if should_skip(name, table, version, year, policy=OVERWRITE_POLICY):
            continue

        try:
            db = mario.parse_eora(path=str(path), multi_region=True)
        except Exception as exc:
            print(f'[skip] parse failed: {type(exc).__name__}: {exc}')
            continue

        # Convert the configured GHG flows into a single electricity-sector export.
        db.calc_ghg(profile='eora', inplace=True)
        efs = emission_factors(
            db, 'Sector', p['labels_list'], ghg_unit='Gg CO2eq'
        )
        export_efs(efs, name, table, version, year)

## EMERGING

This section runs the EMERGING dataset with MARIO's inverse-based compute method, then exports the electricity-sector footprint results for the configured year.

In [2]:
# EMERGING is run with the inverse method before parsing the database.
mario.set_compute_method('inverse')

p = properties['emerging']
name, table = p['name'], p['table']

for version in p['versions']:
    for year in p['years']:
        path = db_path(cfg, base, 'EMERGING', year=year)
        print(f'\n=== {name} {version} {table} {year} ===')

        if should_skip(name, table, version, year, policy=OVERWRITE_POLICY):
            continue

        try:
            db = mario.parse_emerging(path=str(path), year=year)
        except Exception as exc:
            print(f'[skip] parse failed: {type(exc).__name__}: {exc}')
            continue

        # The EMERGING profile returns Mt CO2eq before later unit harmonisation.
        # db.calc_ghg(profile='emerging', inplace=True)
        # efs = emission_factors(
        #     db, 'Sector', p['labels_list'], ghg_unit='Mt CO2eq'
        # )
        # export_efs(efs, name, table, version, year)


=== EMERGING 1.0 IOT 2017 ===
INFO Parser: reading EMERGING bundle global_mrio_2017.mat.
INFO Parser: reading EMERGING CO2 file EMERGING_CO2_2017.mat.
INFO Parser: EMERGING parsed with 245 regions, 134 sectors, 735 final-demand columns and 7 satellite rows.
INFO Metadata: initialized.


In [7]:
# db.calc_ghg(profile='emerging', inplace=True)
# efs = emission_factors(
#     db, 'Sector', p['labels_list'], ghg_unit='Mt CO2eq'
# )
export_efs(efs, name, table, version, year)

-> exported /Users/lorenzorinaldi/Library/CloudStorage/OneDrive-SharedLibraries-PolitecnicodiMilano/DENG-SESAM - Documenti/c-Research/b-Models/MARIO/Models/2026_JIE_Citterio_CFe/export/EMERGING_IOT_1.0_2017.csv


PosixPath('/Users/lorenzorinaldi/Library/CloudStorage/OneDrive-SharedLibraries-PolitecnicodiMilano/DENG-SESAM - Documenti/c-Research/b-Models/MARIO/Models/2026_JIE_Citterio_CFe/export/EMERGING_IOT_1.0_2017.csv')

## GLORIA

GLORIA exposes electricity information in both `Activity` and `Commodity` dimensions. This section computes GHGs for both views, renames the active dimension to `Sector`, and concatenates the results into one export table.

In [ ]:
p = properties['gloria']
name, table = p['name'], p['table']

for version in p['versions']:
    for year in p['years']:
        path = db_path(cfg, base, 'GLORIA', year=year)
        print(f'\n=== {name} {version} {table} {year} ===')

        if should_skip(name, table, version, year, policy=OVERWRITE_POLICY):
            continue

        try:
            db = mario.parse_gloria(path=str(path), year=year)
        except Exception as exc:
            print(f'[skip] parse failed: {type(exc).__name__}: {exc}')
            continue

        db.calc_ghg(profile='gloria', inplace=True)

        # GLORIA stores relevant electricity labels under two dimensions. Rename
        # each active dimension to `Sector` before combining the results.
        efs = pd.concat([
            emission_factors(db, level, labels).rename(columns={level: 'Sector'})
            for level, labels in p['labels_list'].items()
        ], axis=0, ignore_index=True)

        export_efs(efs, name, table, version, year)

## Merge exported CSV files

Run this step after any database sections you executed above. It reproduces the logic in `merge.py` and writes `export/combined_results.csv` with harmonised `kg/EUR` units.

In [ ]:
%run merge.py